# 05 - Q4 scenario: tighten or relax promised delivery times

The brief asks what happens to GMV and repeat rate if we move the promise on specific lanes. Two findings from earlier notebooks shape what this notebook does:

1. **B2 found no measurable 90-day repeat-rate gap** between OnTime and Delayed first-orders (42.41% vs 42.07%). The repeat-rate channel for the GMV impact is effectively zero in this data, so we do not extrapolate a repeat lift the cohort comparison did not show.
2. **Two lanes are extreme outliers** at the current promise: Delhivery -> Jaipur and Delhivery -> Lucknow run at ~86% delayed, vs ~28-30% baseline. The promise change conversation centres on these lanes plus a handful of other heavy-late ones.

What this notebook produces:
- Per-lane on-time rate at the **current** promise vs counterfactual rates at **+1d / +2d / +3d** (relax) and **-1d** (tighten).
- Delayed-GMV reduction at each scenario - the visible "customer experience" channel: support load, fee credits, brand.
- A short read of the table with carrier-level recommendations.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PROC = Path('..') / 'data' / 'processed'
orders_enr = pd.read_parquet(PROC / 'orders_enriched.parquet')
ships_enr  = pd.read_parquet(PROC / 'shipments_enriched.parquet')

len(orders_enr), len(ships_enr)

(100000, 91994)

## Build the scenario flags

`delivery_delay_days = delivered_at - promised_delivery_date`. In this dataset the `OnTime` label means `delay <= -1` (delivered at least one day before the promised date); `Late_1_2d` covers `delay in {0, 1}`. Cross-checked against the raw status field. So the scenario rules are:

- **Current** promise: on-time iff `delay <= -1` (matches the `OnTime` label).
- **Relax +1d**: promise moves out by a day, on-time iff `delay <= 0`.
- **Relax +2d**: on-time iff `delay <= 1`.
- **Relax +3d**: on-time iff `delay <= 2` (catches the Late_3_5d head).
- **Tighten -1d**: on-time iff `delay <= -2`.

Lost shipments (no `delivered_at`) stay "not on-time" in every scenario - widening the promise does not recover a parcel that never arrived. InTransit shipments are excluded since the outcome is unknown.

In [2]:
# evaluable shipments = everything except still-in-transit
ships = ships_enr[ships_enr['delivery_status'] != 'InTransit'].copy()
delay = ships['delivery_delay_days']

ships['ontime_now'] = (delay <= -1).fillna(False)
ships['ontime_p1']  = (delay <=  0).fillna(False)
ships['ontime_p2']  = (delay <=  1).fillna(False)
ships['ontime_p3']  = (delay <=  2).fillna(False)
ships['ontime_m1']  = (delay <= -2).fillna(False)

# Sanity: ontime_now should equal the labelled status truth from notebook 02/03
assert (ships['ontime_now'] == (ships['delivery_status'] == 'OnTime')).all()

ships = ships.merge(orders_enr[['order_id', 'order_gmv']], on='order_id', how='left')

print('evaluable shipments:', len(ships))
print()
print('marketplace on-time rate at each scenario:')
for col, label in [('ontime_m1', 'tighten -1d'),
                   ('ontime_now', 'current    '),
                   ('ontime_p1', 'relax +1d  '),
                   ('ontime_p2', 'relax +2d  '),
                   ('ontime_p3', 'relax +3d  ')]:
    print(f'  {label}: {ships[col].mean():.4f}')

evaluable shipments: 86951

marketplace on-time rate at each scenario:
  tighten -1d: 0.3343
  current    : 0.7324
  relax +1d  : 0.9145
  relax +2d  : 0.9669
  relax +3d  : 0.9848


## Lane-level scenario table

Group by (carrier, ship_to_city). For each lane, report current on-time rate plus the counterfactual rates and the delayed-GMV under each scenario.

In [3]:
def delayed_gmv(group, ontime_col):
    return group.loc[~group[ontime_col], 'order_gmv'].sum()

lane_rows = []
for (carrier, city), g in ships.groupby(['carrier', 'ship_to_city']):
    lane_rows.append({
        'carrier': carrier,
        'ship_to_city': city,
        'shipments': len(g),
        'gmv': g['order_gmv'].sum(),
        'ontime_now': g['ontime_now'].mean(),
        'ontime_p1' : g['ontime_p1'].mean(),
        'ontime_p2' : g['ontime_p2'].mean(),
        'ontime_p3' : g['ontime_p3'].mean(),
        'ontime_m1' : g['ontime_m1'].mean(),
        'delayed_gmv_now': delayed_gmv(g, 'ontime_now'),
        'delayed_gmv_p1' : delayed_gmv(g, 'ontime_p1'),
        'delayed_gmv_p2' : delayed_gmv(g, 'ontime_p2'),
        'delayed_gmv_p3' : delayed_gmv(g, 'ontime_p3'),
        'delayed_gmv_m1' : delayed_gmv(g, 'ontime_m1'),
    })
lanes = pd.DataFrame(lane_rows).sort_values('delayed_gmv_now', ascending=False).reset_index(drop=True)

# Top 10 lanes by today's delayed GMV - the headline targets
show = lanes.head(10).copy()
rate_cols = ['ontime_now', 'ontime_p1', 'ontime_p2', 'ontime_p3', 'ontime_m1']
show[rate_cols] = show[rate_cols].round(3)
gmv_cols = ['gmv', 'delayed_gmv_now', 'delayed_gmv_p1', 'delayed_gmv_p2', 'delayed_gmv_p3', 'delayed_gmv_m1']
for c in gmv_cols:
    show[c] = show[c].round(0).astype('int64')
show

,carrier,ship_to_city,shipments,gmv,ontime_now,ontime_p1,ontime_p2,ontime_p3,ontime_m1,delayed_gmv_now,delayed_gmv_p1,delayed_gmv_p2,delayed_gmv_p3,delayed_gmv_m1
0,Ekart,Kolkata,1824,59794762,0.186,0.364,0.626,0.883,0.085,50306433,41379695,25819845,9496822,55990574
1,Delhivery,Mumbai,3982,132800570,0.730,0.940,0.989,0.995,0.317,47144655,10374958,2012697,591864,93668986
2,Delhivery,Jaipur,1465,48225352,0.133,0.271,0.504,0.762,0.046,42300534,35138648,26251059,13563137,46282809
3,Delhivery,Delhi,3606,119842012,0.735,0.941,0.988,0.995,0.322,41728724,10006581,1703415,217029,83893148
4,Delhivery,Bangalore,3199,109783810,0.725,0.935,0.986,0.994,0.324,40351466,10688616,2124091,799361,78703346
5,Delhivery,Chennai,3199,111375188,0.719,0.939,0.988,0.996,0.308,40287331,9766589,1536902,487432,79125091
6,Delhivery,Lucknow,1384,43896340,0.134,0.274,0.521,0.763,0.040,39283108,33891754,24431558,12237809,42355817
7,BlueDart,Mumbai,3085,103719133,0.729,0.942,0.988,0.993,0.323,35398781,7022409,1265149,491423,72267761
8,Delhivery,Pune,2777,96112578,0.724,0.944,0.992,0.997,0.308,34300094,7751757,934907,195858,70234338
9,BlueDart,Delhi,2699,90407152,0.718,0.935,0.989,0.995,0.322,34059503,8230489,1340937,461294,63914436


## The two broken lanes

Delhivery -> Jaipur and Delhivery -> Lucknow run at roughly 86% delayed today. Walk those two through every scenario.

In [4]:
broken = lanes[(lanes['carrier'] == 'Delhivery')
               & (lanes['ship_to_city'].isin(['Jaipur', 'Lucknow']))]

for _, r in broken.iterrows():
    print(f"{r['carrier']} -> {r['ship_to_city']}  ({int(r['shipments']):,} shipments, GMV Rs {int(r['gmv']):,})")
    print(f"  on-time today : {r['ontime_now']:.3f}")
    print(f"  on-time +1d   : {r['ontime_p1']:.3f}   (delayed GMV drops {int(r['delayed_gmv_now']-r['delayed_gmv_p1']):,})")
    print(f"  on-time +2d   : {r['ontime_p2']:.3f}   (delayed GMV drops {int(r['delayed_gmv_now']-r['delayed_gmv_p2']):,})")
    print(f"  on-time +3d   : {r['ontime_p3']:.3f}   (delayed GMV drops {int(r['delayed_gmv_now']-r['delayed_gmv_p3']):,})")
    print(f"  on-time -1d   : {r['ontime_m1']:.3f}")
    print()

Delhivery -> Jaipur  (1,465 shipments, GMV Rs 48,225,352)
  on-time today : 0.133
  on-time +1d   : 0.271   (delayed GMV drops 7,161,886)
  on-time +2d   : 0.504   (delayed GMV drops 16,049,475)
  on-time +3d   : 0.762   (delayed GMV drops 28,737,397)
  on-time -1d   : 0.046

Delhivery -> Lucknow  (1,384 shipments, GMV Rs 43,896,340)
  on-time today : 0.134
  on-time +1d   : 0.274   (delayed GMV drops 5,391,354)
  on-time +2d   : 0.521   (delayed GMV drops 14,851,550)
  on-time +3d   : 0.763   (delayed GMV drops 27,045,299)
  on-time -1d   : 0.040



Most of the delay mass on these two lanes sits in `Late_1_2d`. That is what the +1d / +2d widening eats.

In [5]:
broken_ships = ships[(ships['carrier'] == 'Delhivery')
                     & (ships['ship_to_city'].isin(['Jaipur', 'Lucknow']))]
print(pd.crosstab(broken_ships['ship_to_city'],
                  broken_ships['delivery_delay_days'].fillna('Lost'),
                  margins=True))

delivery_delay_days  -4.0  -3.0  -2.0  -1.0   0.0   1.0   2.0   3.0   4.0   5.0   Lost   All
ship_to_city                                                                                
Jaipur                  2     7    59   127   202   341   378   263    66    13      7  1465
Lucknow                 2    14    40   129   194   342   335   243    71    10      4  1384
All                     4    21    99   256   396   683   713   506   137    23     11  2849


## Marketplace-level uplift if we widen the broken lanes only

A targeted policy: widen the promise by +1 or +2 days on the two broken lanes only, leave every other lane on the current 2-day promise. This is the change we would actually recommend rather than a global widening.

In [6]:
broken_mask = ((ships['carrier'] == 'Delhivery')
               & (ships['ship_to_city'].isin(['Jaipur', 'Lucknow'])))

scenarios = {}
scenarios['current']                 = ships['ontime_now']
scenarios['widen_broken_lanes_+1d']  = np.where(broken_mask, ships['ontime_p1'], ships['ontime_now'])
scenarios['widen_broken_lanes_+2d']  = np.where(broken_mask, ships['ontime_p2'], ships['ontime_now'])
scenarios['widen_broken_lanes_+3d']  = np.where(broken_mask, ships['ontime_p3'], ships['ontime_now'])
scenarios['widen_all_lanes_+1d']     = ships['ontime_p1']
scenarios['widen_all_lanes_+2d']     = ships['ontime_p2']
scenarios['tighten_all_lanes_-1d']   = ships['ontime_m1']

rows = []
for name, ontime in scenarios.items():
    delayed = ~np.asarray(ontime).astype(bool)
    rows.append({
        'scenario': name,
        'ontime_rate': float(np.asarray(ontime).mean()),
        'delayed_gmv': float(ships.loc[delayed, 'order_gmv'].sum()),
    })
summary = pd.DataFrame(rows)
summary['delayed_gmv_vs_today'] = summary['delayed_gmv'] - summary.loc[0, 'delayed_gmv']
summary['ontime_rate_vs_today'] = summary['ontime_rate'] - summary.loc[0, 'ontime_rate']
summary.round(4)

,scenario,ontime_rate,delayed_gmv,delayed_gmv_vs_today,ontime_rate_vs_today
0,current,0.7324,9.799463e+08,0.000000e+00,0.0000
1,widen_broken_lanes_+1d,0.7369,9.673930e+08,-1.255324e+07,0.0046
2,widen_broken_lanes_+2d,0.7448,9.490453e+08,-3.090102e+07,0.0124
3,widen_broken_lanes_+3d,0.7530,9.241636e+08,-5.578270e+07,0.0206
4,widen_all_lanes_+1d,0.9145,3.013353e+08,-6.786110e+08,0.1821
5,widen_all_lanes_+2d,0.9669,1.108053e+08,-8.691410e+08,0.2345
6,tighten_all_lanes_-1d,0.3343,2.059663e+09,1.079717e+09,-0.3981


## Read of the table

A few things stand out once the numbers are correct:

1. **The current `OnTime` label is unusually strict.** It requires delivery *at least one day before* the promised date. The 15,831 shipments labelled `Late_1_2d` with delay=0 actually arrived on the promised day, which most customers and most marketplaces would call on-time. Re-labelling delay=0 as OnTime (the +1d widening row) would lift the marketplace on-time rate from 73.24% to 91.45% with no operational change. Worth checking with the Logistics lead whether this is a deliberate choice or a definitional accident.

2. **Targeted widening on Delhivery -> Jaipur and Delhivery -> Lucknow only** moves marketplace on-time rate by 0.5 pp at +1d, 1.2 pp at +2d, 2.1 pp at +3d. Even +3d only takes those two lanes to 76% on-time. They are not a "promise problem", they are a capacity problem. Widening the promise hides the symptom but does not fix it. The real intervention on those two lanes is **switching carrier** (InHouse runs at 7.4% delayed marketwide; Delhivery at 33.2%).

3. **Global widening** is mechanically large but cosmetic. +1d alone takes the marketplace from 73% to 91% on-time. It is a relabelling exercise, not an operational gain.

4. **Tightening by 1 day** flips ~40 pp of currently-on-time GMV into "delayed". Given Q2's null repeat-rate finding (no retention payoff to being faster), there is no upside to do this.

Caveat to flag in the readout: this is the **visible delayed-GMV** channel. Actual GMV is not lost when a delivery slips - the order still ships and the customer pays. What changes is fee credits, support volume, and brand perception. Per Q2, the repeat-rate channel does not show up in this dataset.

Recommendation summary:
- Switch carrier on Delhivery -> Jaipur and Delhivery -> Lucknow (root cause).
- Audit the OnTime label definition; aligning with industry-standard "on or before promise day" is a free 18 pp on the headline metric.
- Hold the 2-day promise on every other lane; do not tighten.

## Save outputs for the dashboard

In [7]:
lanes.to_parquet(PROC / 'q4_lane_scenarios.parquet', index=False)
summary.to_parquet(PROC / 'q4_marketplace_summary.parquet', index=False)

for name in ['q4_lane_scenarios', 'q4_marketplace_summary']:
    p = PROC / f'{name}.parquet'
    print(f'{name:30s} {p.stat().st_size/1024:.1f} KB')

q4_lane_scenarios              12.3 KB
q4_marketplace_summary         3.9 KB
